In [3]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [4]:
# ==========================================
# 1. LOAD DATA (already preprocessed, deduplicated, and split)
# ==========================================
TRAIN_FILE = "TrainingDataset.csv"
VAL_FILE = "ValidationDataset.csv"
TEST_FILE = "TestingDataset.csv"
TEXT_COL = 0
LABEL_COL = 1

train_df = pd.read_csv( TRAIN_FILE, header=None)
val_df = pd.read_csv( VAL_FILE, header=None)
test_df = pd.read_csv( TEST_FILE, header=None)

X_train, y_train = train_df[TEXT_COL], train_df[LABEL_COL]
X_val, y_val = val_df[TEXT_COL], val_df[LABEL_COL]
X_test, y_test = test_df[TEXT_COL], test_df[LABEL_COL]

print("Dataset Sizes Loaded:")
print(f" - Train Set:      {len(X_train)} samples")
print(f" - Validation Set: {len(X_val)} samples")
print(f" - Test Set:       {len(X_test)} samples\n")


Dataset Sizes Loaded:
 - Train Set:      108786 samples
 - Validation Set: 14510 samples
 - Test Set:       21757 samples



In [5]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
MAX_VOCAB_SIZE = 30000
MAX_SEQ_LEN = 50
EMBEDDING_DIM = 300
BATCH_SIZE = 64            # fixed; see note at the tuning section below
SEARCH_EPOCHS = 15         # shorter budget per config during the grid search
FINAL_PATIENCE = 4
FASTTEXT_VECTORS_PATH = None  # set to None to train FastText from scratch

In [6]:
X_train_text = train_df[TEXT_COL].astype(str)
X_val_text = val_df[TEXT_COL].astype(str)
X_test_text = test_df[TEXT_COL].astype(str)

print("Dataset Sizes Loaded:")
print(f" - Train Set:      {len(X_train_text)} samples")
print(f" - Validation Set: {len(X_val_text)} samples")
print(f" - Test Set:       {len(X_test_text)} samples\n")

Dataset Sizes Loaded:
 - Train Set:      108786 samples
 - Validation Set: 14510 samples
 - Test Set:       21757 samples



In [7]:
# ==========================================
# 2. LABEL ENCODING
# ==========================================
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(train_df[LABEL_COL])
y_val_int = label_encoder.transform(val_df[LABEL_COL])
y_test_int = label_encoder.transform(test_df[LABEL_COL])

num_classes = len(label_encoder.classes_)
y_train = to_categorical(y_train_int, num_classes=num_classes)
y_val = to_categorical(y_val_int, num_classes=num_classes)
y_test = to_categorical(y_test_int, num_classes=num_classes)

print(f"Classes ({num_classes}): {list(label_encoder.classes_)}\n")

Classes (10): ['اقتصاد', 'تكنولوجيا', 'خواطر واقتباسات', 'دين', 'رياضه', 'سيارت', 'سياسه', 'صحه', 'طهي', 'علوم']



In [8]:
# ==========================================
# 3. TOKENIZATION & SEQUENCE PADDING
# ==========================================
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")

vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

In [9]:
import numpy as np
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 99.6 MB/s eta 0:00:00


In [10]:
# ==========================================
# 4. FASTTEXT EMBEDDING MATRIX
# ==========================================
def load_pretrained_fasttext(path, word_index, vocab_size, embedding_dim):
    embeddings_index = {}
    with open(path, "r", encoding="utf-8", newline="\n", errors="ignore") as f:
        first_line = f.readline().split()
        if len(first_line) != 2:
            f.seek(0)
        for line in f:
            values = line.rstrip().split(" ")
            word = values[0]
            if word in word_index and word_index[word] < vocab_size:
                embeddings_index[word] = np.asarray(values[1:], dtype="float32")

    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    hits = 0
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[idx] = vector
            hits += 1
    print(f"FastText coverage: {hits}/{vocab_size} vocab words found "
          f"({hits / vocab_size:.1%})\n")
    return embedding_matrix


def train_custom_fasttext(texts, word_index, vocab_size, embedding_dim):
    from gensim.models import FastText

    tokenized_texts = [t.split() for t in texts]
    ft_model = FastText(
        sentences=tokenized_texts,
        vector_size=embedding_dim,
        window=5,
        min_count=2,
        workers=4,
        epochs=10,
    )
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    hits = 0
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        if word in ft_model.wv:
            embedding_matrix[idx] = ft_model.wv[word]
            hits += 1
    print(f"Custom FastText coverage: {hits}/{vocab_size} vocab words "
          f"({hits / vocab_size:.1%})\n")
    return embedding_matrix


if FASTTEXT_VECTORS_PATH is not None:
    embedding_matrix = load_pretrained_fasttext(
        FASTTEXT_VECTORS_PATH, tokenizer.word_index, vocab_size, EMBEDDING_DIM
    )
else:
    embedding_matrix = train_custom_fasttext(
        X_train_text, tokenizer.word_index, vocab_size, EMBEDDING_DIM
    )

Custom FastText coverage: 29999/30000 vocab words (100.0%)



In [11]:
# ==========================================
# 5. MODEL BUILDER
# ==========================================
def build_model(lstm_units, dropout_rate, learning_rate):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=EMBEDDING_DIM,
            weights=[embedding_matrix],
            trainable=False,
        ),
        Bidirectional(LSTM(lstm_units, return_sequences=False)),
        Dropout(dropout_rate),
        Dense(64, activation="relu"),
        Dropout(dropout_rate * 0.6),   # lighter second dropout, scaled with the main rate
        Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [12]:
# ==========================================
# 6. HYPERPARAMETER TUNING (manual grid, selected on validation macro-F1)
# ==========================================
# Matches the tuning discipline used for the SVM baseline: search on
# train+validation only, never touch test until the final model is chosen.
#
# A manual grid loop is used here (rather than KerasTuner) to keep the search
# transparent and dependency-light for a term-paper baseline. Batch size is
# left fixed at 64 rather than added to the grid, since each additional grid
# dimension multiplies total training time and batch size tends to have a
# smaller effect on final macro-F1 than the parameters below. Extend
# BATCH_SIZE_GRID below if you have the compute budget for it.
LSTM_UNITS_GRID = [64, 128, 256]
DROPOUT_GRID = [0.3, 0.5]
LR_GRID = [1e-3, 5e-4]

best_val_f1_macro = -1.0
best_config = None
best_model = None

print("Starting hyperparameter grid search (selecting on validation macro-F1)...\n")
for units in LSTM_UNITS_GRID:
    for dropout_rate in DROPOUT_GRID:
        for lr in LR_GRID:
            print(f"--- Config: units={units}, dropout={dropout_rate}, lr={lr} ---")
            model = build_model(units, dropout_rate, lr)

            early_stop = EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
            )
            model.fit(
                X_train_pad, y_train,
                validation_data=(X_val_pad, y_val),
                batch_size=BATCH_SIZE,
                epochs=SEARCH_EPOCHS,
                callbacks=[early_stop],
                verbose=0,
            )

            val_pred_int = np.argmax(model.predict(X_val_pad, verbose=0), axis=1)
            val_f1_macro = f1_score(y_val_int, val_pred_int, average="macro")
            print(f"    Validation Macro-F1: {val_f1_macro:.4f}\n")

            if val_f1_macro > best_val_f1_macro:
                best_val_f1_macro = val_f1_macro
                best_config = {"lstm_units": units, "dropout": dropout_rate, "learning_rate": lr}
                best_model = model

print("Best Hyperparameters Found:")
print(best_config)
print(f"Best Validation Macro-F1: {best_val_f1_macro:.4f}\n")

# Continue training the winning config a bit longer with a more patient
# early-stopping criterion, in case SEARCH_EPOCHS cut it off early.
early_stop_final = EarlyStopping(
    monitor="val_loss",
    patience=FINAL_PATIENCE,
    restore_best_weights=True,
)
best_model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    batch_size=BATCH_SIZE,
    epochs=SEARCH_EPOCHS + 20,
    initial_epoch=SEARCH_EPOCHS,
    callbacks=[early_stop_final],
    verbose=1,
)

Starting hyperparameter grid search (selecting on validation macro-F1)...

--- Config: units=64, dropout=0.3, lr=0.001 ---
    Validation Macro-F1: 0.8781

--- Config: units=64, dropout=0.3, lr=0.0005 ---
    Validation Macro-F1: 0.8836

--- Config: units=64, dropout=0.5, lr=0.001 ---
    Validation Macro-F1: 0.8782

--- Config: units=64, dropout=0.5, lr=0.0005 ---
    Validation Macro-F1: 0.8799

--- Config: units=128, dropout=0.3, lr=0.001 ---
    Validation Macro-F1: 0.8816

--- Config: units=128, dropout=0.3, lr=0.0005 ---
    Validation Macro-F1: 0.8848

--- Config: units=128, dropout=0.5, lr=0.001 ---
    Validation Macro-F1: 0.8784

--- Config: units=128, dropout=0.5, lr=0.0005 ---
    Validation Macro-F1: 0.8861

--- Config: units=256, dropout=0.3, lr=0.001 ---
    Validation Macro-F1: 0.8842

--- Config: units=256, dropout=0.3, lr=0.0005 ---
    Validation Macro-F1: 0.8877

--- Config: units=256, dropout=0.5, lr=0.001 ---
    Validation Macro-F1: 0.8843

--- Config: units=256,

In [13]:
# ==========================================
# 7. VALIDATION SET EVALUATION (final chosen model)
# ==========================================
y_val_pred_int = np.argmax(best_model.predict(X_val_pad, verbose=0), axis=1)
val_f1_macro = f1_score(y_val_int, y_val_pred_int, average="macro")
val_f1_weighted = f1_score(y_val_int, y_val_pred_int, average="weighted")

print("==========================================")
print("VALIDATION SET RESULTS (final model)")
print("==========================================")
print(f"Validation Macro-F1:    {val_f1_macro:.4f}")
print(f"Validation Weighted-F1: {val_f1_weighted:.4f}\n")

VALIDATION SET RESULTS (final model)
Validation Macro-F1:    0.8870
Validation Weighted-F1: 0.8881



In [14]:
# ==========================================
# 8. FINAL TEST SET EVALUATION (once, on held-out test set)
# ==========================================
y_test_pred_int = np.argmax(best_model.predict(X_test_pad, verbose=0), axis=1)
test_f1_macro = f1_score(y_test_int, y_test_pred_int, average="macro")
test_f1_weighted = f1_score(y_test_int, y_test_pred_int, average="weighted")

print("==========================================")
print("FINAL UNSEEN TEST SET RESULTS")
print("==========================================")
print(f"Test Macro-F1:    {test_f1_macro:.4f}")
print(f"Test Weighted-F1: {test_f1_weighted:.4f}\n")

print("Full Classification Report:")
print(classification_report(
    y_test_int, y_test_pred_int, target_names=label_encoder.classes_
))

print("Confusion Matrix (rows=true, cols=predicted):")
cm = confusion_matrix(y_test_int, y_test_pred_int)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print(cm_df, "\n")

FINAL UNSEEN TEST SET RESULTS
Test Macro-F1:    0.8875
Test Weighted-F1: 0.8885

Full Classification Report:
                 precision    recall  f1-score   support

         اقتصاد       0.86      0.92      0.89      2239
      تكنولوجيا       0.95      0.86      0.90      2328
خواطر واقتباسات       0.87      0.86      0.86      2447
            دين       0.88      0.88      0.88      1957
          رياضه       0.97      0.96      0.97      2327
          سيارت       0.95      0.94      0.95      2214
          سياسه       0.91      0.89      0.90      2437
            صحه       0.80      0.86      0.83      2099
            طهي       0.95      0.92      0.93      1719
           علوم       0.75      0.78      0.76      1990

       accuracy                           0.89     21757
      macro avg       0.89      0.89      0.89     21757
   weighted avg       0.89      0.89      0.89     21757

Confusion Matrix (rows=true, cols=predicted):
                 اقتصاد  تكنولوجيا  خواطر وا

In [16]:
predicted_class_names = label_encoder.inverse_transform(y_test_pred_int)
actual_class_names = label_encoder.inverse_transform(y_test_int)

# Create a DataFrame to display test tweets with their predicted classes
results_df = pd.DataFrame({
    'Tweet': X_test.values,
    'Actual_Class': actual_class_names,
    'Predicted_Class': predicted_class_names
})

results_df.to_csv('predicted_tweets.csv', index=False)
print("DataFrame exported to 'predicted_tweets.csv'")

DataFrame exported to 'predicted_tweets.csv'


In [15]:
# ==========================================
# 9. SAVING MODEL, TOKENIZER, LABEL ENCODER
# ==========================================
best_model.save("fasttext_bilstm_model.keras")
joblib.dump(tokenizer, "fasttext_bilstm_tokenizer.joblib")
joblib.dump(label_encoder, "fasttext_bilstm_label_encoder.joblib")
joblib.dump(best_config, "fasttext_bilstm_best_config.joblib")
print("Saved model, tokenizer, label encoder, and best config to disk.")

Saved model, tokenizer, label encoder, and best config to disk.
